# SPS Self-Specialization — Demo Test

A clean demonstration of deterministic Stage 1 self-specialization.

Flow: clone → install → simple tests → growth boundary → manually add S0 → show only S0 → request float specialization → show new S1 → manually execute S1 → add a new rule → test new capability → reject invalid type → run the canonical self-specialization demo.

## 1. Clone latest `main` and install dependencies

In [ ]:
%cd /content
!rm -rf self-specialization
!git clone --branch main --single-branch https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip install -q -r requirements.txt pytest
!git rev-parse --short HEAD

## 2. Run simple tests

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 3. Check allowed/rejected growth rules

In [ ]:
from specialization.type_specialization_rules import is_type_specialization_allowed

allowed = is_type_specialization_allowed(
    'multiply', ['int', 'int'], 'int',
    'multiply', ['float', 'float'], 'float'
)
rejected = is_type_specialization_allowed(
    'multiply', ['int', 'int'], 'int',
    'add', ['int', 'int'], 'int'
)

print('int multiplication -> float multiplication:', 'ALLOWED' if allowed else 'REJECTED')
print('multiplication -> addition:', 'ALLOWED' if rejected else 'REJECTED')
assert allowed
assert not rejected
print('Boundary checks passed.')

## 4. Manually add the initial S0 capability

In [ ]:
import shutil
from specialization import Capability, CapabilityRegistry

storage = CapabilityRegistry.default_storage_dir()
if storage.exists():
    shutil.rmtree(storage)

registry = CapabilityRegistry.persistent()
INTEGER_SOURCE = '''def execute(a: int, b: int) -> int:
    return a * b
'''

integer_mul = Capability.create(
    'IntegerMultiplication',
    '1.0',
    'S0',
    ['int', 'int'],
    'int',
    INTEGER_SOURCE,
)
registry.register(integer_mul)

print('Manually added:')
print(f'  {integer_mul.name} [{integer_mul.state}]')
print('6 * 7 =', integer_mul.execute(6, 7))

## 5. Check capabilities before specialization

At this point only the original programmer-defined S0 capability should exist.

In [ ]:
capabilities_before = registry.all()
print(f'Total capabilities: {len(capabilities_before)}')
for cap in capabilities_before:
    print(f'  {cap.name} [{cap.state}] | input={cap.input_types} | output={cap.output_type}')

assert [cap.name for cap in capabilities_before] == ['IntegerMultiplication']
assert capabilities_before[0].state == 'S0'

## 6. Manually request FloatMultiplication

This is the self-specialization event. No AI model is used. The existing integer multiplication capability is replicated, the separate type-specialization rules are checked, and a new S1 capability is created if the transformation is allowed.

In [ ]:
from specialization import CapabilityDispatcher, EvolutionEngine, Verifier

dispatcher = CapabilityDispatcher(registry, EvolutionEngine(registry, Verifier()))

value, float_mul = dispatcher.execute(
    'multiply',
    2.5,
    4.0,
    ('FloatMultiplication', 'float', [(2.5, 4.0, 10.0), (-2.5, 4.0, -10.0)]),
)

print(f'Created: {float_mul.name} [{float_mul.state}]')
print('Result from generated capability:', value)

## 7. Check capabilities after specialization

The new capability should now be visible in the registry.

In [ ]:
capabilities_after = registry.all()
print(f'Total capabilities: {len(capabilities_after)}')
for cap in sorted(capabilities_after, key=lambda c: (c.created_at, c.id)):
    print(f'  {cap.name} [{cap.state}] | input={cap.input_types} | output={cap.output_type} | parent={cap.parent_id}')

assert registry.get('IntegerMultiplication').state == 'S0'
assert registry.get('FloatMultiplication').state == 'S1'
assert len(capabilities_after) == 3

## 8. Manually execute the new FloatMultiplication capability

In [ ]:
float_cap = registry.get('FloatMultiplication')
print('FloatMultiplication [S1]')
print('3.5 * 2.0 =', float_cap.execute(3.5, 2.0))
print('10.0 * 1.5 =', float_cap.execute(10.0, 1.5))
assert float_cap.execute(3.5, 2.0) == 7.0

## 9. Final capability hierarchy and generated source

In [ ]:
general = registry.get(float_cap.parent_id)
print(f'{general.name} [{general.state}]')
for child in registry.children(general.id):
    print(f'  └── {child.name} [{child.state}]')

print('\nGenerated source code:')
print(float_cap.source_code)

assert general.name == 'SerializeCapability'
assert [c.name for c in registry.children(general.id)] == ['IntegerMultiplication', 'FloatMultiplication']

## 10. Add a new valid rule and test the created capability

This is a runtime-only Colab experiment: the repository files are not changed. We add one new allowed type contract, then evolve the existing `FloatMultiplication` capability into a new S1 capability using the added rule.

New rule: `float × float → int × float`, while preserving the multiplication operation.

In [ ]:
import specialization.type_specialization_rules as rules
import specialization.type_specialization as type_specialization
from specialization.type_specialization_rules import TypeSpecializationRule

# Add a new rule only in this running Colab session.
new_rule = TypeSpecializationRule(
    operation='multiply',
    source_types=('float', 'float'),
    source_output_type='float',
    target_types=('int', 'float'),
    target_output_type='float',
    target_python_types=('int', 'float'),
    target_python_output_type='float',
)
rules.TYPE_SPECIALIZATION_RULES = tuple(rules.TYPE_SPECIALIZATION_RULES) + (new_rule,)

# type_specialization.py imported these functions directly, so patch its
# module-level references for this runtime-only experiment.
type_specialization.get_type_specialization_rule = rules.get_type_specialization_rule
type_specialization.is_type_specialization_allowed = rules.is_type_specialization_allowed

generated = dispatcher.evolution_engine.evolve(
    parent_id=general.id,
    target_name='MixedNumericMultiplication',
    input_types=['int', 'float'],
    output_type='float',
    cases=[(2, 4.5, 9.0), (-3, 2.0, -6.0), (0, 8.25, 0.0)],
    source_capability_id=float_cap.id,
)

print(f'Created: {generated.name} [{generated.state}]')
print('Generated source:')
print(generated.source_code)
print('2 * 4.5 =', generated.execute(2, 4.5))

assert generated.name == 'MixedNumericMultiplication'
assert generated.state == 'S1'
assert generated.execute(2, 4.5) == 9.0
assert rules.get_type_specialization_rule('multiply', ['float', 'float'], 'float', ['int', 'float'], 'float') is not None
print('New valid-rule test passed: generated, verified, activated, and executed.')

## 11. Check an invalid type and confirm it is rejected

This intentionally requests `StringMultiplication` without adding a corresponding rule. The correct result is rejection, with no S1 capability registered.

In [ ]:
invalid_allowed = rules.is_type_specialization_allowed(
    'multiply', ['int', 'int'], 'int',
    'multiply', ['str', 'str'], 'str'
)
print('int multiplication -> string multiplication:', 'ALLOWED' if invalid_allowed else 'REJECTED')
assert not invalid_allowed

before_names = {cap.name for cap in registry.all()}
invalid_result = dispatcher.evolution_engine.evolve(
    parent_id=general.id,
    target_name='StringMultiplication',
    input_types=['str', 'str'],
    output_type='str',
    cases=[('a', 'b', 'ab')],
    source_capability_id=integer_mul.id,
)

print(f'Invalid request result: {invalid_result.name} [{invalid_result.state}]')
print('Failure event:')
for event in invalid_result.events:
    if event.event == 'FAILED':
        print(f'  {event.detail}')

after_names = {cap.name for cap in registry.all()}
assert invalid_result.state == 'FAILED'
assert 'StringMultiplication' not in after_names
assert after_names == before_names
print('Invalid-type test passed: request was rejected and no S1 capability was created.')

## 12. Canonical Self-Specialization Demo Test

Run the repository's main demo last. This confirms the same S0 → S0-C → specialization → S1 → persistence/reload flow as a standalone executable test.

In [ ]:
%cd /content/self-specialization
!SPS_DEMO_RESET=1 PYTHONPATH=. python experiments/self_specialization_demo.py